In [ ]:
#Model Training 

# Import necessary libraries
import os
import gc
import numpy as np
import pydicom
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Configuration settings
IMG_SIZE = (128, 128)  # Resize images to 128x128 to reduce memory usage
BATCH_SIZE = 16        # Small batch size to fit RTX 4050's 6GB memory
EPOCHS = 20            # Maximum epochs, with early stopping to reduce training time
AUTO = tf.data.AUTOTUNE  # Optimize data pipeline performance

# Function to load and preprocess a single DICOM image
def load_dicom(path):
    """
    Loads a DICOM file, applies windowing, normalizes pixel values, and resizes the image.
    
    Args:
        path: Tensor or string path to the DICOM file
    
    Returns:
        Processed image as a numpy array, or a zero tensor if loading fails
    """
    try:
        # Convert tensor path to string if necessary
        path = path.numpy().decode('utf-8') if isinstance(path, tf.Tensor) else path
        dicom = pydicom.dcmread(path, force=True)
        if not hasattr(dicom, 'pixel_array'):
            return np.zeros((*IMG_SIZE, 1), dtype=np.float32)
        img = dicom.pixel_array.astype(np.float32)
        # Apply brain-specific windowing (center=40, width=80)
        center = dicom.get('WindowCenter', 40)
        width = dicom.get('WindowWidth', 80)
        img = np.clip(img, center - width / 2, center + width / 2)
        # Normalize to [0, 1]
        img = (img - img.min()) / (img.max() - img.min() + 1e-7)
        # Resize to target size
        img = cv2.resize(img, IMG_SIZE)
        return img[..., np.newaxis]  # Add channel dimension
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return np.zeros((*IMG_SIZE, 1), dtype=np.float32)  # Return default tensor on failure

# Function to create a TensorFlow dataset
def create_dataset(paths, labels=None):
    """
    Creates a TensorFlow dataset for efficient data loading.
    
    Args:
        paths: List of file paths to DICOM images
        labels: List of corresponding labels (0 for normal, 1 for hemorrhage), or None for inference
    
    Returns:
        A batched and prefetched tf.data.Dataset
    """
    def _process_path(path):
        img = tf.py_function(load_dicom, [path], tf.float32)
        img.set_shape((*IMG_SIZE, 1))
        return img

    if labels is not None:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        ds = ds.map(lambda p, l: (_process_path(p), l), num_parallel_calls=AUTO)
        ds = ds.filter(lambda x, l: tf.math.reduce_sum(x) > 0)  # Filter out all-zero images
    else:
        ds = tf.data.Dataset.from_tensor_slices(paths)
        ds = ds.map(_process_path, num_parallel_calls=AUTO)
        ds = ds.filter(lambda x: tf.math.reduce_sum(x) > 0)  # Filter out all-zero images
    
    return ds.batch(BATCH_SIZE).prefetch(AUTO)

# Function to build the model
def build_model():
    """
    Builds a lightweight CNN model using MobileNetV2 for binary classification.
    
    Returns:
        Compiled Keras model
    """
    base_model = MobileNetV2(input_shape=(*IMG_SIZE, 3), weights='imagenet', include_top=False)
    base_model.trainable = False  # Freeze base model
    
    model = models.Sequential([
        layers.Conv2D(3, (3, 3), padding='same', input_shape=(*IMG_SIZE, 1)),  # Adapt 1-channel to 3-channel
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Function to apply data augmentation
def augment_dataset(ds):
    """
    Applies data augmentation to the training dataset.
    
    Args:
        ds: TensorFlow dataset
    
    Returns:
        Augmented dataset
    """
    augmentation = models.Sequential([
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1)
    ])
    return ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=AUTO)

# Main training function
def train_model(normal_path, hemorrhage_path):
    """
    Trains the model on normal and hemorrhage DICOM images and saves it.
    
    Args:
        normal_path: Path to normal DICOM directory
        hemorrhage_path: Path to hemorrhage DICOM directory
    
    Returns:
        Trained model and training history
    """
    # Collect file paths and labels
    normal_files = [(os.path.join(r, f), 0) for r, _, fs in os.walk(normal_path) for f in fs if f.endswith('.dcm')]
    hemo_files = [(os.path.join(r, f), 1) for r, _, fs in os.walk(hemorrhage_path) for f in fs if f.endswith('.dcm')]
    all_files = normal_files + hemo_files
    paths, labels = zip(*all_files)
    
    print(f"Normal files found: {len(normal_files)}")
    print(f"Hemorrhage files found: {len(hemo_files)}")
    
    # Split data into training and validation sets
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        paths, labels, test_size=0.1, stratify=labels, random_state=42
    )
    
    # Create datasets
    train_ds = create_dataset(train_paths, train_labels)
    val_ds = create_dataset(val_paths, val_labels)
    train_ds = augment_dataset(train_ds)
    
    # Build and train the model
    model = build_model()
    early_stopping = callbacks.EarlyStopping(patience=5, monitor='val_accuracy', restore_best_weights=True)
    reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
    
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    
    # Save the trained model
    model.save('hemorrhage_model.h5')
    print("Model saved as 'hemorrhage_model.h5'")
    
    # Store validation data for evaluation
    global val_p, val_l
    val_p, val_l = val_paths, val_labels
    
    return model, history

# Evaluation function
def evaluate_model(model):
    """
    Evaluates the model on the validation set.
    
    Args:
        model: Trained Keras model
    """
    val_ds = create_dataset(val_p, val_l)
    preds = model.predict(val_ds)
    preds_binary = (preds > 0.5).astype(int)
    print("\nClassification Report:")
    print(classification_report(val_l, preds_binary, target_names=['Normal', 'Hemorrhage']))

# Plotting function
def plot_history(history):
    """
    Plots training and validation accuracy.
    
    Args:
        history: Training history object
    """
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

# Execute the script
normal_path =  r"C:\Users\Atharva Badgujar\OneDrive\Desktop\Hemorrhage new\LSTM model\Normal"
hemorrhage_path =r"C:\Users\Atharva Badgujar\OneDrive\Desktop\Hemorrhage new\LSTM model\HEMORRHAGES CT AI\Hemorrhage" 

print("Starting model training...")
model, history = train_model(normal_path, hemorrhage_path)
print("Evaluating model performance...")
evaluate_model(model)
print("Plotting training history...")
plot_history(history)

# Clean up memory
tf.keras.backend.clear_session()
gc.collect()
print("Training complete.")

In [ ]:
#Evaluation and saving Patiient Resports

# Import necessary libraries
import os
import gc
import numpy as np
import pydicom
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Configuration settings
IMG_SIZE = (128, 128)  # Resize images to 128x128 to reduce memory usage
BATCH_SIZE = 16        # Small batch size to fit RTX 4050's 6GB memory
EPOCHS = 20            # Maximum epochs, with early stopping to reduce training time
AUTO = tf.data.AUTOTUNE  # Optimize data pipeline performance

# Function to load and preprocess a single DICOM image
def load_dicom(path):
    """
    Loads a DICOM file, applies windowing, normalizes pixel values, and resizes the image.
    
    Args:
        path: Tensor or string path to the DICOM file
    
    Returns:
        Processed image as a numpy array, or a zero tensor if loading fails
    """
    try:
        # Convert tensor path to string if necessary
        path = path.numpy().decode('utf-8') if isinstance(path, tf.Tensor) else path
        dicom = pydicom.dcmread(path, force=True)
        if not hasattr(dicom, 'pixel_array'):
            return np.zeros((*IMG_SIZE, 1), dtype=np.float32)
        img = dicom.pixel_array.astype(np.float32)
        # Apply brain-specific windowing (center=40, width=80)
        center = dicom.get('WindowCenter', 40)
        width = dicom.get('WindowWidth', 80)
        img = np.clip(img, center - width / 2, center + width / 2)
        # Normalize to [0, 1]
        img = (img - img.min()) / (img.max() - img.min() + 1e-7)
        # Resize to target size
        img = cv2.resize(img, IMG_SIZE)
        return img[..., np.newaxis]  # Add channel dimension
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return np.zeros((*IMG_SIZE, 1), dtype=np.float32)  # Return default tensor on failure

# Function to create a TensorFlow dataset
def create_dataset(paths, labels=None):
    """
    Creates a TensorFlow dataset for efficient data loading.
    
    Args:
        paths: List of file paths to DICOM images
        labels: List of corresponding labels (0 for normal, 1 for hemorrhage), or None for inference
    
    Returns:
        A batched and prefetched tf.data.Dataset
    """
    def _process_path(path):
        img = tf.py_function(load_dicom, [path], tf.float32)
        img.set_shape((*IMG_SIZE, 1))
        return img

    if labels is not None:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        ds = ds.map(lambda p, l: (_process_path(p), l), num_parallel_calls=AUTO)
        ds = ds.filter(lambda x, l: tf.math.reduce_sum(x) > 0)  # Filter out all-zero images
    else:
        ds = tf.data.Dataset.from_tensor_slices(paths)
        ds = ds.map(_process_path, num_parallel_calls=AUTO)
        ds = ds.filter(lambda x: tf.math.reduce_sum(x) > 0)  # Filter out all-zero images
    
    return ds.batch(BATCH_SIZE).prefetch(AUTO)

# Function to build the model
def build_model():
    """
    Builds a lightweight CNN model using MobileNetV2 for binary classification.
    
    Returns:
        Compiled Keras model
    """
    base_model = MobileNetV2(input_shape=(*IMG_SIZE, 3), weights='imagenet', include_top=False)
    base_model.trainable = False  # Freeze base model
    
    model = models.Sequential([
        layers.Conv2D(3, (3, 3), padding='same', input_shape=(*IMG_SIZE, 1)),  # Adapt 1-channel to 3-channel
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Function to apply data augmentation
def augment_dataset(ds):
    """
    Applies data augmentation to the training dataset.
    
    Args:
        ds: TensorFlow dataset
    
    Returns:
        Augmented dataset
    """
    augmentation = models.Sequential([
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1)
    ])
    return ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=AUTO)

# Main training function
def train_model(normal_path, hemorrhage_path):
    """
    Trains the model on normal and hemorrhage DICOM images and saves it.
    
    Args:
        normal_path: Path to normal DICOM directory
        hemorrhage_path: Path to hemorrhage DICOM directory
    
    Returns:
        Trained model and training history
    """
    # Collect file paths and labels
    normal_files = [(os.path.join(r, f), 0) for r, _, fs in os.walk(normal_path) for f in fs if f.endswith('.dcm')]
    hemo_files = [(os.path.join(r, f), 1) for r, _, fs in os.walk(hemorrhage_path) for f in fs if f.endswith('.dcm')]
    all_files = normal_files + hemo_files
    paths, labels = zip(*all_files)
    
    print(f"Normal files found: {len(normal_files)}")
    print(f"Hemorrhage files found: {len(hemo_files)}")
    
    # Split data into training and validation sets
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        paths, labels, test_size=0.1, stratify=labels, random_state=42
    )
    
    # Create datasets
    train_ds = create_dataset(train_paths, train_labels)
    val_ds = create_dataset(val_paths, val_labels)
    train_ds = augment_dataset(train_ds)
    
    # Build and train the model
    model = build_model()
    early_stopping = callbacks.EarlyStopping(patience=5, monitor='val_accuracy', restore_best_weights=True)
    reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
    
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    
    # Save the trained model
    model.save('hemorrhage_model.h5')
    print("Model saved as 'hemorrhage_model.h5'")
    
    # Store validation data for evaluation
    global val_p, val_l
    val_p, val_l = val_paths, val_labels
    
    return model, history

# Evaluation function
def evaluate_model(model):
    """
    Evaluates the model on the validation set.
    
    Args:
        model: Trained Keras model
    """
    val_ds = create_dataset(val_p, val_l)
    preds = model.predict(val_ds)
    preds_binary = (preds > 0.5).astype(int)
    print("\nClassification Report:")
    print(classification_report(val_l, preds_binary, target_names=['Normal', 'Hemorrhage']))

# Plotting function
def plot_history(history):
    """
    Plots training and validation accuracy.
    
    Args:
        history: Training history object
    """
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

# Execute the script
normal_path =  r"C:\Users\Atharva Badgujar\OneDrive\Desktop\Hemorrhage new\LSTM model\Normal"
hemorrhage_path =r"C:\Users\Atharva Badgujar\OneDrive\Desktop\Hemorrhage new\LSTM model\HEMORRHAGES CT AI\Hemorrhage" 

print("Starting model training...")
model, history = train_model(normal_path, hemorrhage_path)
print("Evaluating model performance...")
evaluate_model(model)
print("Plotting training history...")
plot_history(history)

# Clean up memory
tf.keras.backend.clear_session()
gc.collect()
print("Training complete.")